# Threshold Tuning: FAR vs FRR
Notebook ini digunakan untuk bereksperimen dengan berbagai nilai *Threshold* ($\tau$) dalam pipeline Face Recognition untuk menganalisis performa sistem melalui metrik *False Accept Rate* (FAR) dan *False Reject Rate* (FRR).

In [ ]:
import sys
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.append(project_root)

from backend.app.services.face_service import FaceService
from backend.app.utils.config import KNOWN_FACES_DIR, FACE_DB_PATH

service = FaceService(
    known_faces_dir=str(KNOWN_FACES_DIR),
    db_path=str(FACE_DB_PATH)
)

if not service.database:
    print("Database is empty! Build the database first via API or face_service.py")
else:
    print(f"Database loaded with {len(service.database)} identities.")

### Simulasi Distribusi Similarity Score
Di sini kita dapat mensimulasikan sekumpulan skor *cosine similarity* untuk kasus *Genuine* (wajah sama) dan *Imposter* (wajah berbeda) lalu menghitung FAR dan FRR berdasarkan nilai threshold yang bervariasi dari 0.3 hingga 0.6.

In [ ]:
# Simulasi data (Anda dapat menggantinya dengan perhitungan aktual dari dataset validasi)
np.random.seed(42)
genuine_scores = np.random.normal(0.7, 0.1, 1000) # Skor tinggi (mendekati 1.0)
imposter_scores = np.random.normal(0.2, 0.15, 1000) # Skor rendah

# Batasi agar tidak lebih dari 1.0
genuine_scores = np.clip(genuine_scores, 0, 1.0)
imposter_scores = np.clip(imposter_scores, 0, 1.0)

thresholds = np.linspace(0.3, 0.6, 50)
fars = []
frrs = []

for tau in thresholds:
    # False Accept: Imposter yang skornya >= tau
    far = np.sum(imposter_scores >= tau) / len(imposter_scores)
    # False Reject: Genuine yang skornya < tau
    frr = np.sum(genuine_scores < tau) / len(genuine_scores)
    
    fars.append(far)
    frrs.append(frr)

plt.figure(figsize=(10, 6))
plt.plot(thresholds, fars, label="FAR (False Accept Rate)", color='red')
plt.plot(thresholds, frrs, label="FRR (False Reject Rate)", color='blue')
plt.xlabel("Threshold ($\tau$)")
plt.ylabel("Rate")
plt.title("FAR vs FRR Threshold Tuning Curve")
plt.grid(True)
plt.legend()

# Mencari EER (Equal Error Rate) 
eer_idx = np.argmin(np.abs(np.array(fars) - np.array(frrs)))
optimal_tau = thresholds[eer_idx]
eer_val = fars[eer_idx]

plt.axvline(x=optimal_tau, color='green', linestyle='--', label=f'Optimal $\tau$ ~ {optimal_tau:.2f}')
plt.legend()
plt.show()

print(f"Estimated Optimal Threshold (EER): {optimal_tau:.3f}")